In [4]:
#!pip install -q jarvis-tools
from jarvis.core.atoms import Atoms
from jarvis.db.figshare import data
from jarvis.db.jsonutils import loadjson, dumpjson

def get_crystal_string_t(atoms):
    lengths = atoms.lattice.abc  # structure.lattice.parameters[:3]
    angles = atoms.lattice.angles
    atom_ids = atoms.elements
    frac_coords = atoms.frac_coords

    crystal_str = \
    " ".join(["{0:.1f}".format(x) for x in lengths]) + "\n" + \
    " ".join([str(int(x)) for x in angles]) + "\n" + \
    "\n".join([
        str(t) + "\n" + " ".join([
            "{0:.2f}".format(x) for x in c
        ]) for t,c in zip(atom_ids, frac_coords)
    ])

    # crystal_str = atoms_describer(atoms) + "\n*\n" + crystal_str
    return crystal_str
    
def make_alpaca_json(dataset=[], prop="Tc_supercon"):
    mem = []
    for i in dataset:
        if i[prop] != "na":
            atoms = Atoms.from_dict(i["atoms"])
            info = {}
            info["instruction"] = (
                "Below is a description of a superconductor material."
            )
            info["input"] = (
                "The chemical formula is "
                + atoms.composition.reduced_formula
                + ". The  "
                + prop
                + " value is "
                + str(round(i[prop], 3))
                + ". The spacegroup is "
                + i["spg_number"]
                + "."
                + " Generate atomic structure description with lattice lengths, angles, coordinates and atom types."
            )
            info["response"] = get_crystal_string_t(atoms)
            mem.append(info)
    return mem

In [12]:
dft_3d = data("dft_3d")
print(len(dft_3d))
dataset = make_alpaca_json(dataset=dft_3d, prop="Tc_supercon")
dumpjson(data=dataset, filename="alpaca_Tc_supercon.json")

dataset[0]["instruction"], dataset[0]["input"], dataset[0]["response"]
len(dataset) 

from datasets import load_dataset
dataset = load_dataset(
    "json", data_files="alpaca_Tc_supercon.json", split="train"
)

alpaca_prompt = """Below is a description of a superconductor material. Write a response that appropriately completes the request.

### Input:
{}

### Response:
{}"""



Obtaining 3D dataset 76k ...
Reference:https://www.nature.com/articles/s41524-020-00440-1
Other versions:https://doi.org/10.6084/m9.figshare.6815699
Loading the zipfile...
Loading completed.
75993


Generating train split: 1058 examples [00:00, 124315.71 examples/s]


TypeError: 'LlamaTokenizerFast' object is not subscriptable

In [13]:
tokenizer

LlamaTokenizerFast(name_or_path='_unsloth_sentencepiece_temp/unsloth_llama-2-7b-bnb-4bit', vocab_size=32000, model_max_length=4096, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<unk>'}, clean_up_tokenization_spaces=False),  added_tokens_decoder={
	0: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}

In [17]:
from functools import partial
partial_func = partial(formatting_prompts_func, tokenizer=tokenizer)

In [18]:
def formatting_prompts_func(examples, tokenizer = None):

    #instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["response"]
    texts = []
    for input, output in zip(inputs, outputs):
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        text = alpaca_prompt.format(input, output) + tokenizer.eos_token
        texts.append(text)

    return {
        "text": texts,
    }
    
dataset = dataset.map(
    partial_func,
    batched=True,
)
# dataset = formatting_prompts_func(dataset)

Map: 100%|████████████████| 1058/1058 [00:00<00:00, 101741.88 examples/s]


In [19]:
dataset["text"][0]

'Below is a description of a superconductor material. Write a response that appropriately completes the request.\n\n### Input:\nThe chemical formula is SrB6. The  Tc_supercon value is 0.005. The spacegroup is 221. Generate atomic structure description with lattice lengths, angles, coordinates and atom types.\n\n### Response:\n4.2 4.2 4.2\n90 90 90\nSr\n0.00 0.00 0.00\nB\n0.20 0.50 0.50\nB\n0.50 0.50 0.80\nB\n0.50 0.50 0.20\nB\n0.50 0.20 0.50\nB\n0.50 0.80 0.50\nB\n0.80 0.50 0.50</s>'

In [3]:
len(dataset)

1058

In [9]:
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
import torch
max_seq_length = 2048  # Choose any! We auto support RoPE Scaling internally!
dtype = None  # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = (
    True  # Use 4bit quantization to reduce memory usage. Can be False.
)

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/mistral-7b-bnb-4bit",
    "unsloth/mistral-7b-instruct-v0.2-bnb-4bit",
    "unsloth/llama-2-7b-bnb-4bit",
    "unsloth/llama-2-13b-bnb-4bit",
    "unsloth/codellama-34b-bnb-4bit",
    "unsloth/tinyllama-bnb-4bit",
]  # More models at https://huggingface.co/unsloth

nm = "unsloth/mistral-7b-bnb-4bit"
model_string = 'meta-llama/Llama-2-7b-hf'
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_string,  # Choose ANY! eg teknium/OpenHermes-2.5-Mistral-7B
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)
# from peft import (
#     LoraConfig, 
#     get_peft_model, 
#     prepare_model_for_kbit_training
# )

# lora_config = LoraConfig(
#         r=16,
#         lora_alpha=16,
#         lora_dropout=0.05,
#         bias="none",
#         task_type="CAUSAL_LM",
#     )

# model = get_peft_model(model, lora_config)
# model.print_trainable_parameters()

model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,  # Supports any, but = 0 is optimized
    bias="none",  # Supports any, but = "none" is optimized
    use_gradient_checkpointing=True,
    random_state=3407,
    use_rslora=False,  # We support rank stabilized LoRA
    loftq_config=None,  # And LoftQ
)
model.print_trainable_parameters()

Unsloth: You passed in `meta-llama/Llama-2-7b-hf` and `load_in_4bit = True`.
We shall load `unsloth/llama-2-7b-bnb-4bit` for 4x faster loading.
/home/jipengsun/.conda/envs/llm/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Unused kwargs: ['quant_method']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.


==((====))==  Unsloth: Fast Llama patching release 2024.4
   \\   /|    GPU: Quadro RTX 8000. Max memory: 44.481 GB. Platform = Linux.
O^O/ \_/ \    Pytorch: 2.1.1+cu121. CUDA = 7.5. CUDA Toolkit = 12.1.
\        /    Bfloat16 = FALSE. Xformers = 0.0.23. FA = False.
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth


You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers
Unsloth 2024.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


trainable params: 39,976,960 || all params: 6,778,392,576 || trainable%: 0.589770503135875


In [10]:
tokenizer.eos_token

'</s>'

In [20]:
#model.save_pretrained("llama_model_t")

In [25]:
#model, tokenizer = FastLanguageModel.from_pretrained("/home/jipengsun/atom-gen/m-saved_model")

Unused kwargs: ['quant_method']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.


==((====))==  Unsloth: Fast Mistral patching release 2024.3
   \\   /|    GPU: Quadro RTX 8000. Max memory: 44.481 GB. Platform = Linux.
O^O/ \_/ \    Pytorch: 2.2.1. CUDA = 7.5. CUDA Toolkit = 12.1.
\        /    Bfloat16 = FALSE. Xformers = 0.0.25. FA = False.
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth


In [10]:
trainer = SFTTrainer(
    model=mode,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,  # Can make training 5x faster for short sequences.
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        overwrite_output_dir=True,
        # max_steps = 60,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        num_train_epochs=5,
        report_to="none",
    ),
)
trainer_stats = trainer.train()
model.save_pretrained("llama_model_m")

Map (num_proc=2): 100%|█████████████████| 1058/1058 [00:00<00:00, 1998.43 examples/s]
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 1,058 | Num Epochs = 5
O^O/ \_/ \    Batch size per device = 2 | Gradient Accumulation steps = 4
\        /    Total batch size = 8 | Total steps = 660
 "-____-"     Number of trainable parameters = 41,943,040


Step,Training Loss
1,1.650900
2,1.539000
3,1.414200


KeyboardInterrupt: 

In [55]:
dataset[0]["text"]

'Below is a description of a superconductor material..\n\n### Instruction:\nBelow is a description of a superconductor material.\n\n### Input:\nThe chemical formula is SrB6The  Tc_supercon is 0.005. The spacegroup is 221. Generate atomic structure description with lattice lengths, angles, coordinates and atom types.\n\n### Output:\n4.19 4.19 4.19\n90 90 90\nSr 0.000 0.000 0.000\nB 0.203 0.500 0.500\nB 0.500 0.500 0.797\nB 0.500 0.500 0.203\nB 0.500 0.203 0.500\nB 0.500 0.797 0.500\nB 0.797 0.500 0.500</s>'

In [26]:
FastLanguageModel.for_inference(model)  # Enable native 2x faster inference

In [52]:
alpaca_prompt = """Below is a description of a superconductor material..

### Instruction:
{}

### Input:
{}

### Output:
{}"""

def eval_prompts(examples):
    
    instructions = examples["instruction"]
    inputs = examples["input"]
    output =  examples["output"]
    texts = []
    for instruction, input in zip(instructions, inputs):
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        text = alpaca_prompt.format(instruction, input, output) #+  '</s>' #tokenizer.eos_token #'</s>' #tokenizer.eos_token
        texts.append(text)
    return {
        "prompt": texts,
    }
datasett = dataset.map(
        eval_prompts,
        batched=True,
    )

Map: 100%|█| 1058/1058 [00:01<00:00, 736.96 example


In [29]:
input_ch = datasett["prompt"][0]
inputs = tokenizer(
    [
       input_ch
    ],
    return_tensors="pt",
    ).to("cuda")
input_ch

'Below is a description of a superconductor material..\n\n### Instruction:\nBelow is a description of a superconductor material.\n\n### Input:\nThe chemical formula is SrB6The  Tc_supercon is 0.005. The spacegroup is 221. Generate atomic structure description with lattice lengths, angles, coordinates and atom types.\n\n### Output:\n'

In [30]:
outputs = model.generate(**inputs, max_new_tokens=512, use_cache=True)
output = tokenizer.batch_decode(outputs,skip_special_tokens=True, clean_up_tokenization_spaces=True)[0]

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [31]:
output

'Below is a description of a superconductor material..\n\n### Instruction:\nBelow is a description of a superconductor material.\n\n### Input:\nThe chemical formula is SrB6The  Tc_supercon is 0.005. The spacegroup is 221. Generate atomic structure description with lattice lengths, angles, coordinates and atom types.\n\n### Output:\n4.2 4.2 4.2\n90 90 90\nSr\n0.00 0.00 0.00\nB\n0.20 0.50 0.50\nB\n0.50 0.50 0.80\nB\n0.50 0.50 0.20\nB\n0.50 0.20 0.50\nB\n0.50 0.80 0.50\nB\n0.80 0.50 0.50'

In [33]:
tt = output.replace(input_ch, "")
tt

'4.2 4.2 4.2\n90 90 90\nSr\n0.00 0.00 0.00\nB\n0.20 0.50 0.50\nB\n0.50 0.50 0.80\nB\n0.50 0.50 0.20\nB\n0.50 0.20 0.50\nB\n0.50 0.80 0.50\nB\n0.80 0.50 0.50'

In [7]:
inputs = tokenizer(
    [
        alpaca_prompt.format(
            "Below is a description of a superconductor material.",  # instruction
            "The chemical formula is YCI The  Tc_supercon is 6.483. The spacegroup is 12. Generate atomic structure description with lattice lengths, angles, coordinates and atom types.",  # input
            "",  # output - leave this blank for generation!
        )
    ],
    return_tensors="pt",
).to("cuda")
outputs = model.generate(**inputs, max_new_tokens=512, use_cache=True)


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


In [9]:
akpaca = "Below is a description of a superconductor material..\n\n    ### Instruction:\n    Below is a description of a superconductor material.\n        \n    ### Input:\n    The chemical formula is SrB6The  Tc_supercon is 0.005. The spacegroup is 221. Generate atomic structure description with lattice lengths, angles, coordinates and atom types.\n\n    ### Output:\n    </s>"


In [8]:
tokenizer.batch_decode(outputs)

['<s> Below is a description of a superconductor material..\n\n    ### Instruction:\n    Below is a description of a superconductor material.\n        \n    ### Input:\n    The chemical formula is SrB6The  Tc_supercon is 0.005. The spacegroup is 221. Generate atomic structure description with lattice lengths, angles, coordinates and atom types.\n\n    ### Output:\n    </s>же\n\n    ### Solution:\n    The chemical formula is SrB6. The  Tc_supercon is 0.005. The spacegroup is 221. Generate atomic structure description with lattice lengths, angles, coordinates and atom types.\n\n    ### Explanation:\n    The chemical formula is SrB6. The  Tc_supercon is 0.005. The spacegroup is 221. Generate atomic structure description with lattice lengths, angles, coordinates and atom types.\n\n    ### Code:\n    ```\n    #include <iostream>\n    #include <string>\n    #include <vector>\n    #include <map>\n    #include <algorithm>\n    #include <cmath>\n    #include <cstdlib>\n    #include <cstdio>\n  

In [12]:
output = tokenizer.batch_decode(outputs,skip_special_tokens=True, clean_up_tokenization_spaces=True)[0]
output

'Below is a description of a superconductor material..\n\n    ### Instruction:\n    Below is a description of a superconductor material.\n        \n    ### Input:\n    The chemical formula is SrB6The  Tc_supercon is 0.005. The spacegroup is 221. Generate atomic structure description with lattice lengths, angles, coordinates and atom types.\n\n    ### Output:\n    же\n\n    ### Solution:\n    The chemical formula is SrB6. The  Tc_supercon is 0.005. The spacegroup is 221. Generate atomic structure description with lattice lengths, angles, coordinates and atom types.\n\n    ### Explanation:\n    The chemical formula is SrB6. The  Tc_supercon is 0.005. The spacegroup is 221. Generate atomic structure description with lattice lengths, angles, coordinates and atom types.\n\n    ### Code:\n    ```\n    #include <iostream>\n    #include <string>\n    #include <vector>\n    #include <map>\n    #include <algorithm>\n    #include <cmath>\n    #include <cstdlib>\n    #include <cstdio>\n    #includ

In [10]:
tt = output.replace(akpaca, "")

NameError: name 'output' is not defined

In [11]:
output

NameError: name 'output' is not defined

In [76]:
lines = [x for x in tt.split("\n") if len(x) > 0]
lines

['3.58 5.03 6.03',
 '89 89 61',
 'Y 0.750 0.249 0.198',
 'Y 0.250 0.751 0.802',
 'C 0.750 0.571 0.797',
 'C 0.250 0.429 0.203',
 'I 0.750 0.829 0.402',
 'I 0.250 0.171 0.598']

In [80]:
lengths = [float(x) for x in lines[0].split(" ")]
lengths

[3.58, 5.03, 6.03]

In [82]:
angles = [float(x) for x in lines[1].split(" ")]
angles

[89.0, 89.0, 61.0]

In [84]:
species = [line.split(" ")[0] for line in lines[2::]] #['Tm', 'Tm', 'Zn', 'Rh']
species

['Y', 'Y', 'C', 'C', 'I', 'I']

In [86]:
coords = [[float(y) for y in x.split(" ")[1:]] for x in lines[2::]]
coords

[[0.75, 0.249, 0.198],
 [0.25, 0.751, 0.802],
 [0.75, 0.571, 0.797],
 [0.25, 0.429, 0.203],
 [0.75, 0.829, 0.402],
 [0.25, 0.171, 0.598]]

In [87]:
from pymatgen.core import Structure
from pymatgen.core.lattice import Lattice

structure = Structure(
        lattice=Lattice.from_parameters(
            *(lengths + angles)),
        species=species,
        coords=coords, 
        coords_are_cartesian=False,
    )

In [89]:
cif_form = structure.to(fmt="cif")
cif_form

"# generated using pymatgen\ndata_YCI\n_symmetry_space_group_name_H-M   'P 1'\n_cell_length_a   3.58000000\n_cell_length_b   5.03000000\n_cell_length_c   6.03000000\n_cell_angle_alpha   89.00000000\n_cell_angle_beta   89.00000000\n_cell_angle_gamma   61.00000000\n_symmetry_Int_Tables_number   1\n_chemical_formula_structural   YCI\n_chemical_formula_sum   'Y2 C2 I2'\n_cell_volume   94.95076657\n_cell_formula_units_Z   2\nloop_\n _symmetry_equiv_pos_site_id\n _symmetry_equiv_pos_as_xyz\n  1  'x, y, z'\nloop_\n _atom_site_type_symbol\n _atom_site_label\n _atom_site_symmetry_multiplicity\n _atom_site_fract_x\n _atom_site_fract_y\n _atom_site_fract_z\n _atom_site_occupancy\n  Y  Y0  1  0.75000000  0.24900000  0.19800000  1\n  Y  Y1  1  0.25000000  0.75100000  0.80200000  1\n  C  C2  1  0.75000000  0.57100000  0.79700000  1\n  C  C3  1  0.25000000  0.42900000  0.20300000  1\n  I  I4  1  0.75000000  0.82900000  0.40200000  1\n  I  I5  1  0.25000000  0.17100000  0.59800000  1\n"

In [92]:
try:
    _ = Structure.from_str(cif_form, fmt="cif")
except Exception as e:
    print(e)


In [ ]:
    #         outputs.append({
    #             "gen_str": gen_str,
    #             "cif": cif_str,
    #             "model_name": args.model_name,
    #         })

    # df = pd.DataFrame(outputs)
    # df.to_csv(out_path, index=False)

In [36]:
'4.7 4.7 4.7\n59 59 59\nTm\n0.32 0.88 0.96\nTm\n0.82 0.38 0.46\nZn\n0.57 0.13 0.21\nRh\n0.07 0.63 0.71'
print(tokenizer.batch_decode(outputs))
batch = tokenizer(prompt, return_tensors="pt")
batch = {k: v.cuda() for k, v in batch.items()}
batch

{'input_ids': tensor([[    1, 13866,   338,   263,  6139,   310,   263, 21610,  5518, 29889,
           3251,   403,   263,  6139,   310,   278, 27497,   322, 23619,   310,
            278, 24094, 12047,   322,   769,   278,  1543,  1134,   322, 10350,
            363,  1269, 12301,  2629,   278, 24094, 29901,    13]],
        device='cuda:0'),
 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
          1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}

In [26]:
generate_ids = model.generate(
            **batch,
            do_sample=True,
            max_new_tokens=500,
            temperature=0.9, 
            top_p=0.9, 
        )

In [28]:
gen_strs = tokenizer.batch_decode(
            generate_ids, 
            skip_special_tokens=True, 
            clean_up_tokenization_spaces=False
        )
gen_strs

['Below is a description of a bulk material. Generate a description of the lengths and angles of the lattice vectors and then the element type and coordinates for each atom within the lattice:\nThe chemical formula is 2H2C2F8T2M2T2BT2T2I2S2T2T2B2T2P2T2N2P2T2Rh2T2T2T2Rh2Rh2MnT2T2T2SnT2T2T2T2T2T2T2SnMnRhT2RhRhRhMnT2T2T2RhRhRhRhRhT2MnRhT2RhT2RhRhRhRhRhRhT2RhRhT2RhT2RhT2RhT2T2T2T2RhT2RhRhRhRhT2RhRhRhT2RhRhRhT2RhT2RhT2RhRhT2RhRhRhT2RhRhRhT2RhRhRhRhT2RhRhRhT2RhRhRhT2RhRhT2RhRhRhT2RhRhRhT2RhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRh']

In [30]:
gen_strs

['Below is a description of a bulk material. Generate a description of the lengths and angles of the lattice vectors and then the element type and coordinates for each atom within the lattice:\nThe chemical formula is 2H2C2F8T2M2T2BT2T2I2S2T2T2B2T2P2T2N2P2T2Rh2T2T2T2Rh2Rh2MnT2T2T2SnT2T2T2T2T2T2T2SnMnRhT2RhRhRhMnT2T2T2RhRhRhRhRhT2MnRhT2RhT2RhRhRhRhRhRhT2RhRhT2RhT2RhT2RhT2T2T2T2RhT2RhRhRhRhT2RhRhRhT2RhRhRhT2RhT2RhT2RhRhT2RhRhRhT2RhRhRhT2RhRhRhRhT2RhRhRhT2RhRhRhT2RhRhT2RhRhRhT2RhRhRhT2RhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRh']

In [34]:
material_str = gen_strs[0].replace(prompt, "")
material_str

'The chemical formula is 2H2C2F8T2M2T2BT2T2I2S2T2T2B2T2P2T2N2P2T2Rh2T2T2T2Rh2Rh2MnT2T2T2SnT2T2T2T2T2T2T2SnMnRhT2RhRhRhMnT2T2T2RhRhRhRhRhT2MnRhT2RhT2RhRhRhRhRhRhT2RhRhT2RhT2RhT2RhT2T2T2T2RhT2RhRhRhRhT2RhRhRhT2RhRhRhT2RhT2RhT2RhRhT2RhRhRhT2RhRhRhT2RhRhRhRhT2RhRhRhT2RhRhRhT2RhRhT2RhRhRhT2RhRhRhT2RhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRhT2RhRhRhRh'